#  SQLNOIR PROJECT


# Mystery #1 

I want to find the cities where we are sending the most shipments but getting the least money back.

### Section 1 — Count how many orders go to each city

In [ ]:
SELECT
    o.ShipToCity,
    o.ShipToCountry,
    COUNT(o.OrderId) AS TotalOrders
FROM Sales.[Order] AS o
WHERE o.ShipToCity IS NOT NULL
GROUP BY o.ShipToCity, o.ShipToCountry
ORDER BY TotalOrders DESC;

(71 rows affected)

ShipToCity      | ShipToCountry | TotalOrders
----------------+---------------+------------
Rio de Janeiro  | Brazil        | 34         
London          | UK            | 33         
Boise           | USA           | 31         
Sao Paulo       | Brazil        | 31         
Graz            | Austria       | 30         
Cunewalde       | Germany       | 28         
México D.F.     | Mexico        | 28         
Cork            | Ireland       | 19         
Bräcke          | Sweden        | 19         
Luleå           | Sweden        | 18         
Albuquerque     | USA           | 18         
San Cristóbal   | Venezuela     | 18         
Marseille       | France        | 17         
Buenos Aires    | Argentina     | 16         
Oulu            | Finland       | 15         
Frankfurt a.M.  | Germany       | 15         
München         | Germany       | 15         
Toulouse        | France        | 14         
Brandenburg     | Germany       | 14         
Tsawassen       | Canada        | 

Just a basic count grouped by city. I included country too because there might be cities with the same name in different countries. This gives me the shipping volume side of the picture.

### Section 2 — Add the revenue each city actually generated

In [ ]:
SELECT
    o.ShipToCity,
    o.ShipToCountry,
    COUNT(DISTINCT o.OrderId)   AS TotalOrders,
    SUM(od.LineAmount)          AS TotalRevenue
FROM Sales.[Order] AS o
JOIN Sales.OrderDetail AS od 
    ON o.OrderId = od.OrderId
WHERE o.ShipToCity IS NOT NULL
GROUP BY o.ShipToCity, o.ShipToCountry
ORDER BY TotalOrders DESC;

(70 rows affected)

ShipToCity      | ShipToCountry | TotalOrders | TotalRevenue
----------------+---------------+-------------+-------------
Rio de Janeiro  | Brazil        | 34          | 53999.18    
London          | UK            | 33          | 40663.71    
Boise           | USA           | 31          | 115673.39   
Sao Paulo       | Brazil        | 31          | 45786.37    
Graz            | Austria       | 30          | 113236.68   
Cunewalde       | Germany       | 28          | 117483.39   
México D.F.     | Mexico        | 28          | 24073.45    
Bräcke          | Sweden        | 19          | 32555.55    
Cork            | Ireland       | 19          | 57317.39    
Luleå           | Sweden        | 18          | 26968.15    
Albuquerque     | USA           | 18          | 52245.90    
San Cristóbal   | Venezuela     | 18          | 23611.58    
Marseille       | France        | 17          | 23850.95    
Buenos Aires    | Argentina     | 16          | 8119.10     
Oulu            | Finlan

Added the JOIN to OrderDetail and swapped to COUNT DISTINCT on OrderId since one order can have multiple detail rows. Now I can see both how often we ship somewhere and how much we make from it.

### Section 3 — Calculate revenue per order for each city

In [68]:
SELECT
    o.ShipToCity,
    o.ShipToCountry,
    COUNT(DISTINCT o.OrderId) AS TotalOrders,
    SUM(od.LineAmount) AS TotalRevenue,
    SUM(od.LineAmount) / COUNT(DISTINCT o.OrderId) AS RevenuePerOrder
FROM Sales.[Order] AS o
JOIN Sales.OrderDetail AS od 
    ON o.OrderId = od.OrderId
WHERE o.ShipToCity IS NOT NULL
GROUP BY o.ShipToCity, o.ShipToCountry
HAVING COUNT(DISTINCT o.OrderId) >= 5
ORDER BY RevenuePerOrder ASC;

(62 rows affected)

ShipToCity      | ShipToCountry | TotalOrders | TotalRevenue | RevenuePerOrder
----------------+---------------+-------------+--------------+----------------
Barcelona       | Spain         | 5           | 836.70       | 167.34         
Torino          | Italy         | 6           | 1545.70      | 257.6166       
Reims           | France        | 5           | 1480.00      | 296.00         
Helsinki        | Finland       | 7           | 3161.35      | 451.6214       
Mannheim        | Germany       | 7           | 3239.80      | 462.8285       
Warszawa        | Poland        | 7           | 3531.95      | 504.5642       
Buenos Aires    | Argentina     | 16          | 8119.10      | 507.4437       
Elgin           | USA           | 5           | 3063.20      | 612.64         
Cowes           | UK            | 10          | 6146.30      | 614.63         
Aachen          | Germany       | 6           | 3763.21      | 627.2016       
Reggio Emilia   | Italy         | 12          | 7555

A city with 20 orders but only $50 per order is way less valuable than a city with 5 orders at $500 each. The HAVING >= 5 removes cities we have barely shipped to so the results are meaningful.

### Section 4 (Final) — CTE to rank cities by revenue per order

In [87]:
WITH CityValue AS (
    SELECT
        o.ShipToCity,
        o.ShipToCountry,
        COUNT(DISTINCT o.OrderId) AS TotalOrders,
        SUM(od.LineAmount) AS TotalRevenue,
        SUM(od.LineAmount) * 1 / COUNT(DISTINCT o.OrderId) AS RevenuePerOrder
    FROM Sales.[Order] AS o
    JOIN Sales.OrderDetail AS od 
        ON o.OrderId = od.OrderId
    WHERE o.ShipToCity IS NOT NULL
    GROUP BY o.ShipToCity, o.ShipToCountry
    HAVING COUNT(DISTINCT o.OrderId) >= 5
)

SELECT TOP 20
    ShipToCity,
    ShipToCountry,
    TotalOrders,
    TotalRevenue,
    RevenuePerOrder
FROM CityValue
ORDER BY RevenuePerOrder ASC;

(20 rows affected)

ShipToCity    | ShipToCountry | TotalOrders | TotalRevenue | RevenuePerOrder
--------------+---------------+-------------+--------------+----------------
Barcelona     | Spain         | 5           | 836.70       | 167.34         
Torino        | Italy         | 6           | 1545.70      | 257.6166       
Reims         | France        | 5           | 1480.00      | 296.00         
Helsinki      | Finland       | 7           | 3161.35      | 451.6214       
Mannheim      | Germany       | 7           | 3239.80      | 462.8285       
Warszawa      | Poland        | 7           | 3531.95      | 504.5642       
Buenos Aires  | Argentina     | 16          | 8119.10      | 507.4437       
Elgin         | USA           | 5           | 3063.20      | 612.64         
Cowes         | UK            | 10          | 6146.30      | 614.63         
Aachen        | Germany       | 6           | 3763.21      | 627.2016       
Reggio Emilia | Italy         | 12          | 7555.60      | 629.6333       

Orders the cities from worst value to best. The top of this list is cities we are putting shipping effort into but barely making money from. 

# Mystery #2 

My manager mentioned that some employees seem to be handling way more than their fair share of orders. I want to check if the workload is actually spread evenly or if a few people are doing everything while others barely contribute.

### Section 1 — Count orders per employee

In [93]:
SELECT
    e.EmployeeId,
    e.EmployeeFirstName + ' ' + e.EmployeeLastName  AS EmployeeName,
    e.EmployeeTitle,
    COUNT(o.OrderId) AS TotalOrders
FROM HumanResources.Employee AS e
JOIN Sales.[Order] AS o 
    ON e.EmployeeId = o.EmployeeId
GROUP BY e.EmployeeId, e.EmployeeFirstName, e.EmployeeLastName, e.EmployeeTitle
ORDER BY TotalOrders DESC;

(9 rows affected)

EmployeeId | EmployeeName   | EmployeeTitle         | TotalOrders
-----------+----------------+-----------------------+------------
4          | Yael Peled     | Sales Representative  | 156        
3          | Judy Lew       | Sales Manager         | 127        
1          | Sara Davis     | CEO                   | 124        
8          | Maria Cameron  | Sales Representative  | 104        
2          | Don Funk       | Vice President, Sales | 96         
7          | Russell King   | Sales Representative  | 72         
6          | Paul Suurs     | Sales Representative  | 67         
9          | Patricia Doyle | Sales Representative  | 43         
5          | Sven Mortensen | Sales Manager         | 42         
(9 rows)

Here we are just counting how many orders each employee has processed. Ordering highest to lowest already shows me who's doing the most work.

### Section 2 — Add revenue and unique customer count per employee

In [95]:
SELECT
    e.EmployeeFirstName + ' ' + e.EmployeeLastName AS EmployeeName,
    e.EmployeeTitle,
    COUNT(DISTINCT o.OrderId) AS TotalOrders,
    COUNT(DISTINCT o.CustomerId) AS UniqueCustomers,
    SUM(od.LineAmount) AS TotalRevenue
FROM HumanResources.Employee AS e
JOIN Sales.[Order] AS o       
    ON e.EmployeeId = o.EmployeeId
JOIN Sales.OrderDetail AS od  
    ON o.OrderId = od.OrderId
GROUP BY 
    e.EmployeeId, 
    e.EmployeeFirstName, 
    e.EmployeeLastName, 
    e.EmployeeTitle
ORDER BY TotalOrders DESC;

(9 rows affected)

EmployeeName   | EmployeeTitle         | TotalOrders | UniqueCustomers | TotalRevenue
---------------+-----------------------+-------------+-----------------+-------------
Yael Peled     | Sales Representative  | 156         | 75              | 250187.45   
Judy Lew       | Sales Manager         | 127         | 63              | 213051.30   
Sara Davis     | CEO                   | 123         | 65              | 202143.71   
Maria Cameron  | Sales Representative  | 104         | 56              | 133301.03   
Don Funk       | Vice President, Sales | 96          | 59              | 177749.26   
Russell King   | Sales Representative  | 72          | 45              | 141295.99   
Paul Suurs     | Sales Representative  | 67          | 43              | 78198.10    
Patricia Doyle | Sales Representative  | 43          | 29              | 82964.00    
Sven Mortensen | Sales Manager         | 42          | 29              | 75567.75    
(9 rows)

Unique customer count tells a different story than total orders. An employee handling 200 orders for 5 clients is different from handling 200 orders for 80 clients. Both metrics matter for workload fairness.

### Section 3 — Compare each employee to the team average

In [ ]:
SELECT
    e.EmployeeFirstName + ' ' + e.EmployeeLastName AS EmployeeName,
    e.EmployeeTitle,
    COUNT(DISTINCT o.OrderId) AS TotalOrders,
    (
        SELECT AVG(EmpOrders)
        FROM (
            SELECT COUNT(DISTINCT o2.OrderId) AS EmpOrders
            FROM Sales.[Order] AS o2
            GROUP BY o2.EmployeeId
        ) AS AvgCalc
    ) AS TeamAvgOrders,
    COUNT(DISTINCT o.OrderId) - (
        SELECT AVG(EmpOrders)
        FROM (
            SELECT COUNT(DISTINCT o2.OrderId) AS EmpOrders
            FROM Sales.[Order] AS o2
            GROUP BY o2.EmployeeId
        ) AS AvgCalc
    ) AS DiffFromAvg
FROM HumanResources.Employee AS e
JOIN Sales.[Order] AS o ON e.EmployeeId = o.EmployeeId
GROUP BY e.EmployeeId, e.EmployeeFirstName, e.EmployeeLastName, e.EmployeeTitle
ORDER BY DiffFromAvg DESC;

(9 rows affected)

EmployeeName   | EmployeeTitle         | TotalOrders | TeamAvgOrders | DiffFromAvg
---------------+-----------------------+-------------+---------------+------------
Yael Peled     | Sales Representative  | 156         | 92            | 64         
Judy Lew       | Sales Manager         | 127         | 92            | 35         
Sara Davis     | CEO                   | 124         | 92            | 32         
Maria Cameron  | Sales Representative  | 104         | 92            | 12         
Don Funk       | Vice President, Sales | 96          | 92            | 4          
Russell King   | Sales Representative  | 72          | 92            | -20        
Paul Suurs     | Sales Representative  | 67          | 92            | -25        
Patricia Doyle | Sales Representative  | 43          | 92            | -49        
Sven Mortensen | Sales Manager         | 42          | 92            | -50        
(9 rows)

### Section 4 (Final) — CTE with workload flag

In [104]:
WITH WorkloadCheck AS (
    SELECT
        e.EmployeeFirstName + ' ' + e.EmployeeLastName  AS EmployeeName,
        e.EmployeeTitle,
        COUNT(DISTINCT o.OrderId) AS TotalOrders,
        COUNT(DISTINCT o.CustomerId) AS UniqueCustomers,
        ROUND(SUM(od.LineAmount), 2) AS TotalRevenue
    FROM HumanResources.Employee e
    JOIN Sales.[Order] AS o       
        ON e.EmployeeId = o.EmployeeId
    JOIN Sales.OrderDetail AS od  
        ON o.OrderId    = od.OrderId
    GROUP BY e.EmployeeId, e.EmployeeFirstName, e.EmployeeLastName, e.EmployeeTitle
)
SELECT
    EmployeeName,
    EmployeeTitle,
    TotalOrders,
    UniqueCustomers,
    TotalRevenue,
    ROUND(AVG(CAST(TotalOrders AS FLOAT)) OVER (), 1)  AS TeamAvg,
    CASE
        WHEN TotalOrders > AVG(CAST(TotalOrders AS FLOAT)) OVER () * 1.5
            THEN 'Overloaded'
        WHEN TotalOrders < AVG(CAST(TotalOrders AS FLOAT)) OVER () * 0.5
            THEN 'Underutilized'
        ELSE 'Normal'
    END AS WorkloadStatus
FROM WorkloadCheck
ORDER BY TotalOrders DESC;

(9 rows affected)

EmployeeName   | EmployeeTitle         | TotalOrders | UniqueCustomers | TotalRevenue | TeamAvg | WorkloadStatus
---------------+-----------------------+-------------+-----------------+--------------+---------+---------------
Yael Peled     | Sales Representative  | 156         | 75              | 250187.45    | 92.2    | Overloaded    
Judy Lew       | Sales Manager         | 127         | 63              | 213051.30    | 92.2    | Normal        
Sara Davis     | CEO                   | 123         | 65              | 202143.71    | 92.2    | Normal        
Maria Cameron  | Sales Representative  | 104         | 56              | 133301.03    | 92.2    | Normal        
Don Funk       | Vice President, Sales | 96          | 59              | 177749.26    | 92.2    | Normal        
Russell King   | Sales Representative  | 72          | 45              | 141295.99    | 92.2    | Normal        
Paul Suurs     | Sales Representative  | 67          | 43              | 78198.10     | 92.2    

This gives management a quick status column without them having to do math. Identified which employees are carrying too much of the workload and which ones have room for more.

# Mystery #3

We run frequent promotions, but I want to identify products that have never been sold at a discount to determine whether they are consistently popular or simply not selling.

### Section 1 — Look at discount history per product

In [106]:
SELECT
    od.ProductId,
    MIN(od.DiscountPercentage)  AS MinDiscount,
    MAX(od.DiscountPercentage)  AS MaxDiscount,
    COUNT(od.OrderId) AS TimesSold
FROM Sales.OrderDetail AS od
GROUP BY od.ProductId
ORDER BY MaxDiscount ASC;

(77 rows affected)

ProductId | MinDiscount | MaxDiscount | TimesSold
----------+-------------+-------------+----------
15        | 0.000       | 0.050       | 6        
3         | 0.000       | 0.100       | 12       
50        | 0.000       | 0.100       | 10       
49        | 0.000       | 0.150       | 21       
66        | 0.000       | 0.150       | 8        
48        | 0.000       | 0.150       | 6        
73        | 0.000       | 0.200       | 14       
53        | 0.000       | 0.200       | 30       
45        | 0.000       | 0.200       | 14       
37        | 0.000       | 0.200       | 6        
32        | 0.000       | 0.200       | 15       
7         | 0.000       | 0.200       | 29       
65        | 0.000       | 0.200       | 32       
19        | 0.000       | 0.250       | 37       
44        | 0.000       | 0.250       | 24       
1         | 0.000       | 0.250       | 38       
67        | 0.000       | 0.250       | 10       
27        | 0.000       | 0.250       | 9        


If MAX(DiscountPercentage) is 0 for a product, it has never been sold at any discount at all. MIN and MAX together show me the full range of discounts that product has ever seen.

### Section 2 — Join product names and see their sales volume

In [116]:
WITH NeverDiscounted AS (
    SELECT
        p.ProductName,
        MAX(od.DiscountPercentage)                  AS MaxDiscountEver,
        COUNT(od.OrderId)                           AS TimesSold,
        SUM(od.Quantity)                            AS TotalUnitsSold,
        CAST(SUM(od.LineAmount) AS DECIMAL(10,2))   AS TotalRevenue
    FROM Production.Product p
    JOIN Sales.OrderDetail od ON p.ProductId = od.ProductId
    GROUP BY p.ProductId, p.ProductName
)
SELECT TOP 10
    ProductName,
    MaxDiscountEver,
    TimesSold,
    TotalUnitsSold,
    TotalRevenue
FROM NeverDiscounted
ORDER BY MaxDiscountEver ASC, TotalRevenue DESC;

(10 rows affected)

ProductName   | MaxDiscountEver | TimesSold | TotalUnitsSold | TotalRevenue
--------------+-----------------+-----------+----------------+-------------
Product KSZOI | 0.050           | 6         | 122            | 1813.50     
Product BIUDV | 0.100           | 10        | 235            | 3510.00     
Product IMEHJ | 0.100           | 12        | 328            | 3080.00     
Product FPYPN | 0.150           | 21        | 520            | 9500.00     
Product LQMGN | 0.150           | 8         | 239            | 3519.00     
Product MYNXN | 0.150           | 6         | 138            | 1542.75     
Product HMLNI | 0.200           | 29        | 763            | 22464.00    
Product BKGEA | 0.200           | 30        | 722            | 21510.20    
Product XYWBZ | 0.200           | 32        | 745            | 14607.00    
Product NUNAW | 0.200           | 15        | 297            | 9171.20     
(10 rows)

HAVING MAX = 0 filters to only the never-discounted products. Sorting by revenue shows me which of these undiscounted products are actually valuable — those are the ones that probably don't need a discount because people buy them anyway.

### Section 3 — Add category context to see if it is a pattern

In [117]:
SELECT
    c.CategoryName,
    COUNT(DISTINCT p.ProductId)                 AS LeastDiscountedProducts,
    SUM(od.Quantity)                            AS TotalUnitsSold,
    CAST(SUM(od.LineAmount) AS DECIMAL(10,2))   AS TotalRevenue,
    MAX(od.DiscountPercentage)                  AS MaxDiscountInCategory
FROM Production.Category c
JOIN Production.Product p   ON c.CategoryId = p.CategoryId
JOIN Sales.OrderDetail od   ON p.ProductId  = od.ProductId
GROUP BY c.CategoryId, c.CategoryName
ORDER BY MaxDiscountInCategory ASC, TotalRevenue DESC;

(8 rows affected)

CategoryName   | LeastDiscountedProducts | TotalUnitsSold | TotalRevenue | MaxDiscountInCategory
---------------+-------------------------+----------------+--------------+----------------------
Beverages      | 12                      | 9532           | 286526.95    | 0.250                
Dairy Products | 10                      | 9149           | 251330.50    | 0.250                
Meat/Poultry   | 6                       | 4199           | 178188.80    | 0.250                
Confections    | 13                      | 7906           | 177099.10    | 0.250                
Seafood        | 12                      | 7681           | 141623.09    | 0.250                
Condiments     | 12                      | 5298           | 113694.75    | 0.250                
Produce        | 5                       | 2990           | 105268.60    | 0.250                
Grains/Cereals | 7                       | 4562           | 100726.80    | 0.250                
(8 rows)

Grouping by category tells me if the no-discount pattern is category-wide. If a whole category shows up here that is probably a deliberate pricing policy, not just random.

### Section 4 (Final) — Final list of never-discounted products with revenue rank

In [118]:
;WITH LeastDiscounted AS (
    SELECT
        p.ProductName,
        c.CategoryName,
        COUNT(od.OrderId)                           AS TimesSold,
        SUM(od.Quantity)                            AS UnitsSold,
        CAST(SUM(od.LineAmount) AS DECIMAL(10,2))   AS TotalRevenue,
        CAST(AVG(od.UnitPrice)  AS DECIMAL(10,2))   AS AvgSellingPrice,
        MAX(od.DiscountPercentage)                  AS MaxDiscountEver
    FROM Production.Product p
    JOIN Production.Category c  ON p.CategoryId = c.CategoryId
    JOIN Sales.OrderDetail od   ON p.ProductId  = od.ProductId
    GROUP BY p.ProductId, p.ProductName, c.CategoryName
)
SELECT
    ROW_NUMBER() OVER (ORDER BY TotalRevenue DESC)  AS RevenueRank,
    ProductName,
    CategoryName,
    TimesSold,
    UnitsSold,
    TotalRevenue,
    AvgSellingPrice,
    MaxDiscountEver
FROM LeastDiscounted
ORDER BY MaxDiscountEver ASC, RevenueRank;

(77 rows affected)

RevenueRank | ProductName   | CategoryName   | TimesSold | UnitsSold | TotalRevenue | AvgSellingPrice | MaxDiscountEver
------------+---------------+----------------+-----------+-----------+--------------+-----------------+----------------
75          | Product KSZOI | Condiments     | 6         | 122       | 1813.50      | 14.47           | 0.050          
69          | Product BIUDV | Confections    | 10        | 235       | 3510.00      | 14.95           | 0.100          
71          | Product IMEHJ | Condiments     | 12        | 328       | 3080.00      | 9.50            | 0.100          
43          | Product FPYPN | Confections    | 21        | 520       | 9500.00      | 18.48           | 0.150          
68          | Product LQMGN | Condiments     | 8         | 239       | 3519.00      | 15.30           | 0.150          
77          | Product MYNXN | Confections    | 6         | 138       | 1542.75      | 11.90           | 0.150          
16          | Product HMLNI | Produce   

revenue shows which of the never-discounted products are actually the most important to us. Top ranked ones are clearly doing fine without discounts. Bottom ranked ones might benefit from a promotion to boost sales.


# Mystery #4 

I want to know if these big single orders happen more during certain parts of the year. Maybe there is a seasonal pattern we could use for planning.


### Section 1 — Find the total value of each individual order

In [119]:
SELECT
    od.OrderId,
    o.OrderDate,
    ROUND(SUM(od.LineAmount), 2) AS OrderValue,
    COUNT(od.ProductId) AS LineItems
FROM Sales.OrderDetail AS od
JOIN Sales.[Order] AS o 
    ON od.OrderId = o.OrderId
GROUP BY od.OrderId, o.OrderDate
ORDER BY OrderValue DESC;

(830 rows affected)

OrderId | OrderDate  | OrderValue | LineItems
--------+------------+------------+----------
10865   | 2022-02-02 | 17250.00   | 2        
11030   | 2022-04-17 | 16321.90   | 4        
10981   | 2022-03-27 | 15810.00   | 1        
10372   | 2020-12-04 | 12281.20   | 4        
10424   | 2021-01-23 | 11493.20   | 3        
10817   | 2022-01-06 | 11490.70   | 4        
10889   | 2022-02-16 | 11380.00   | 2        
10417   | 2021-01-16 | 11283.20   | 4        
10897   | 2022-02-19 | 10835.24   | 2        
10353   | 2020-11-13 | 10741.60   | 2        
10515   | 2021-04-23 | 10588.50   | 5        
10479   | 2021-03-19 | 10495.60   | 4        
10540   | 2021-05-19 | 10191.70   | 4        
10691   | 2021-10-03 | 10164.80   | 5        
11032   | 2022-04-17 | 8902.50    | 3        
10816   | 2022-01-06 | 8891.00    | 2        
10514   | 2021-04-22 | 8623.45    | 5        
10912   | 2022-02-26 | 8267.40    | 2        
10360   | 2020-11-22 | 7390.20    | 5        
10776   | 2021-12-15 | 6984.50    

Summing LineAmount per OrderId gives me each order's total value. LineItems count shows how many products were in each order. 

### Section 2 — Group orders by quarter and find the average order size

In [120]:
-- using MONTH() and CASE to group into quarters manually
-- then averaging order values per quarter

SELECT
    YEAR(o.OrderDate)                           AS OrderYear,
    CASE
        WHEN MONTH(o.OrderDate) IN (1,2,3)  THEN 'Q1 Jan-Mar'
        WHEN MONTH(o.OrderDate) IN (4,5,6)  THEN 'Q2 Apr-Jun'
        WHEN MONTH(o.OrderDate) IN (7,8,9)  THEN 'Q3 Jul-Sep'
        ELSE                                     'Q4 Oct-Dec'
    END                                         AS Quarter,
    COUNT(DISTINCT o.OrderId)                   AS TotalOrders,
    CAST(AVG(od.LineAmount) AS DECIMAL(10,2))   AS AvgLineValue,
    CAST(SUM(od.LineAmount) AS DECIMAL(10,2))   AS TotalQuarterRevenue
FROM Sales.[Order] o
JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
GROUP BY
    YEAR(o.OrderDate),
    CASE
        WHEN MONTH(o.OrderDate) IN (1,2,3)  THEN 'Q1 Jan-Mar'
        WHEN MONTH(o.OrderDate) IN (4,5,6)  THEN 'Q2 Apr-Jun'
        WHEN MONTH(o.OrderDate) IN (7,8,9)  THEN 'Q3 Jul-Sep'
        ELSE                                     'Q4 Oct-Dec'
    END
ORDER BY OrderYear, Quarter;

(8 rows affected)

OrderYear | Quarter    | TotalOrders | AvgLineValue | TotalQuarterRevenue
----------+------------+-------------+--------------+--------------------
2020      | Q3 Jul-Sep | 70          | 456.42       | 84437.50           
2020      | Q4 Oct-Dec | 82          | 644.82       | 141861.00          
2021      | Q1 Jan-Mar | 92          | 613.61       | 147879.90          
2021      | Q2 Apr-Jun | 93          | 599.25       | 151611.09          
2021      | Q3 Jul-Sep | 103         | 645.23       | 165179.64          
2021      | Q4 Oct-Dec | 120         | 626.92       | 193718.12          
2022      | Q1 Jan-Mar | 182         | 697.44       | 315242.12          
2022      | Q2 Apr-Jun | 88          | 646.57       | 154529.22          
(8 rows)

### Section 3 — Find the top 10 biggest individual orders and what quarter they fell in

In [121]:
SELECT TOP 10
    o.OrderId,
    o.OrderDate,
    CASE
        WHEN MONTH(o.OrderDate) IN (1,2,3)  THEN 'Q1 Jan-Mar'
        WHEN MONTH(o.OrderDate) IN (4,5,6)  THEN 'Q2 Apr-Jun'
        WHEN MONTH(o.OrderDate) IN (7,8,9)  THEN 'Q3 Jul-Sep'
        ELSE                                     'Q4 Oct-Dec'
    END AS Quarter,
    c.CustomerCompanyName,
    CAST(SUM(od.LineAmount) AS DECIMAL(10,2))   AS OrderTotal,
    COUNT(od.ProductId) AS ProductsBought
FROM Sales.[Order] AS o
JOIN Sales.OrderDetail AS od  
    ON o.OrderId    = od.OrderId
JOIN Sales.Customer AS c      
    ON o.CustomerId = c.CustomerId
GROUP BY o.OrderId, o.OrderDate, c.CustomerCompanyName
ORDER BY OrderTotal DESC;

(10 rows affected)

OrderId | OrderDate  | Quarter    | CustomerCompanyName | OrderTotal | ProductsBought
--------+------------+------------+---------------------+------------+---------------
10865   | 2022-02-02 | Q1 Jan-Mar | Customer IRRVL      | 17250.00   | 2             
11030   | 2022-04-17 | Q2 Apr-Jun | Customer LCOUJ      | 16321.90   | 4             
10981   | 2022-03-27 | Q1 Jan-Mar | Customer IBVRG      | 15810.00   | 1             
10372   | 2020-12-04 | Q4 Oct-Dec | Customer WFIZJ      | 12281.20   | 4             
10424   | 2021-01-23 | Q1 Jan-Mar | Customer PVDZC      | 11493.20   | 3             
10817   | 2022-01-06 | Q1 Jan-Mar | Customer GLLAG      | 11490.70   | 4             
10889   | 2022-02-16 | Q1 Jan-Mar | Customer NYUHS      | 11380.00   | 2             
10417   | 2021-01-16 | Q1 Jan-Mar | Customer JMIKW      | 11283.20   | 4             
10897   | 2022-02-19 | Q1 Jan-Mar | Customer FRXZL      | 10835.24   | 2             
10353   | 2020-11-13 | Q4 Oct-Dec | Customer LOLJO    

This shows the actual top 10 biggest orders with the customer name and which quarter it fell in. The CASE just makes the quarter label more readable than just a number.

### Section 4 (Final) — CTE summarizing average order value per quarter

In [122]:
;WITH OrderTotals AS (
    SELECT
        o.OrderId,
        o.OrderDate,
        CASE
            WHEN MONTH(o.OrderDate) IN (1,2,3)  THEN 'Q1 Jan-Mar'
            WHEN MONTH(o.OrderDate) IN (4,5,6)  THEN 'Q2 Apr-Jun'
            WHEN MONTH(o.OrderDate) IN (7,8,9)  THEN 'Q3 Jul-Sep'
            ELSE                                     'Q4 Oct-Dec'
        END                                         AS Quarter,
        CAST(SUM(od.LineAmount) AS DECIMAL(10,2))   AS OrderTotal
    FROM Sales.[Order] o
    JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
    GROUP BY o.OrderId, o.OrderDate
)
SELECT
    Quarter,
    COUNT(OrderId)                              AS TotalOrders,
    CAST(AVG(OrderTotal) AS DECIMAL(10,2))      AS AvgOrderValue,
    CAST(MAX(OrderTotal) AS DECIMAL(10,2))      AS LargestOrder,
    CAST(MIN(OrderTotal) AS DECIMAL(10,2))      AS SmallestOrder
FROM OrderTotals
GROUP BY Quarter
ORDER BY Quarter;

(4 rows affected)

Quarter    | TotalOrders | AvgOrderValue | LargestOrder | SmallestOrder
-----------+-------------+---------------+--------------+--------------
Q1 Jan-Mar | 274         | 1690.23       | 17250.00     | 30.00        
Q2 Apr-Jun | 181         | 1691.38       | 16321.90     | 45.00        
Q3 Jul-Sep | 173         | 1442.87       | 6483.05      | 28.00        
Q4 Oct-Dec | 202         | 1661.28       | 12281.20     | 12.50        
(4 rows)

The CTE calculates each order's total first which avoids trying to nest SUM inside AVG. Then the outer query averages those totals by quarter. It shows which quarter has the highest average order value. We Found which quarters see the biggest individual orders so we can plan inventory and staffing around those peak spending periods.

# Mystery #5 
I want to look at how many products each supplier provides and then see how many orders those products actually generate. Some suppliers might be giving us lots of products that nobody's buying.

### Section 1 — Count products per supplier

In [123]:
SELECT
    s.SupplierId,
    s.SupplierCompanyName,
    COUNT(p.ProductId)      AS ProductsSupplied
FROM Production.Supplier s
JOIN Production.Product AS p 
    ON s.SupplierId = p.SupplierId
GROUP BY s.SupplierId, s.SupplierCompanyName
ORDER BY ProductsSupplied DESC;

(29 rows affected)

SupplierId | SupplierCompanyName | ProductsSupplied
-----------+---------------------+-----------------
7          | Supplier GQRCV      | 5               
12         | Supplier SVIYA      | 5               
8          | Supplier BWGYE      | 4               
2          | Supplier VHQZD      | 4               
3          | Supplier STUAZ      | 3               
4          | Supplier QOVFD      | 3               
1          | Supplier SWRXU      | 3               
6          | Supplier QWUSF      | 3               
14         | Supplier KEREV      | 3               
15         | Supplier NZLIF      | 3               
16         | Supplier UHZRG      | 3               
17         | Supplier QZGUF      | 3               
23         | Supplier ELCRN      | 3               
24         | Supplier JNNES      | 3               
20         | Supplier CIYNM      | 3               
11         | Supplier ZPYVS      | 3               
21         | Supplier XOXZA      | 2               
22         |

Just counting the product catalog size per supplier. 

### Section 2 — Add how many order lines each supplier's products appear in

In [125]:
SELECT
    s.SupplierCompanyName,
    COUNT(DISTINCT p.ProductId) AS ProductsSupplied,
    COUNT(od.OrderId) AS TotalOrderLines,
    COUNT(od.OrderId) / COUNT(DISTINCT p.ProductId) AS OrdersPerProduct
FROM Production.Supplier AS s
JOIN Production.Product AS p   
    ON s.SupplierId = p.SupplierId
JOIN Sales.OrderDetail AS od   
    ON p.ProductId  = od.ProductId
GROUP BY s.SupplierId, s.SupplierCompanyName
ORDER BY OrdersPerProduct ASC;

(29 rows affected)

SupplierCompanyName | ProductsSupplied | TotalOrderLines | OrdersPerProduct
--------------------+------------------+-----------------+-----------------
Supplier FNUXM      | 2                | 27              | 13              
Supplier VHQZD      | 4                | 70              | 17              
Supplier QZGUF      | 3                | 51              | 17              
Supplier QQYEU      | 2                | 34              | 17              
Supplier QOVFD      | 3                | 51              | 17              
Supplier ZRYDZ      | 1                | 18              | 18              
Supplier STUAZ      | 3                | 54              | 18              
Supplier ZPYVS      | 3                | 59              | 19              
Supplier XOXZA      | 2                | 41              | 20              
Supplier UHZRG      | 3                | 65              | 21              
Supplier QWUSF      | 3                | 68              | 22              
Supplier ELC

### Section 3 — Add revenue per product to see financial efficiency

In [126]:
-- same idea but now looking at revenue per product instead of order count
-- a supplier could have high order count but low-value products

SELECT
    s.SupplierCompanyName,
    s.SupplierCountry,
    COUNT(DISTINCT p.ProductId)                                         AS ProductsSupplied,
    COUNT(od.OrderId)                                                   AS TotalOrderLines,
    CAST(SUM(od.LineAmount) AS DECIMAL(10,2))                           AS TotalRevenue,
    CAST(SUM(od.LineAmount) / COUNT(DISTINCT p.ProductId) AS DECIMAL(10,2)) AS RevenuePerProduct
FROM Production.Supplier s
JOIN Production.Product p   ON s.SupplierId = p.SupplierId
JOIN Sales.OrderDetail od   ON p.ProductId  = od.ProductId
GROUP BY s.SupplierId, s.SupplierCompanyName, s.SupplierCountry
ORDER BY RevenuePerProduct ASC;

(29 rows affected)

SupplierCompanyName | SupplierCountry | ProductsSupplied | TotalOrderLines | TotalRevenue | RevenuePerProduct
--------------------+-----------------+------------------+-----------------+--------------+------------------
Supplier FNUXM      | Netherlands     | 2                | 27              | 5901.35      | 2950.68          
Supplier UNAHG      | Brazil          | 1                | 51              | 4782.60      | 4782.60          
Supplier QWUSF      | Japan           | 3                | 68              | 15678.30     | 5226.10          
Supplier XOXZA      | Denmark         | 2                | 41              | 10884.50     | 5442.25          
Supplier QQYEU      | Sweden          | 2                | 34              | 12072.60     | 6036.30          
Supplier ZRYDZ      | France          | 1                | 18              | 6664.75      | 6664.75          
Supplier QZGUF      | Sweden          | 3                | 51              | 21789.80     | 7263.27          
Supplier U

Revenue per product is a stronger than just order count. A product that sells once for $5,000 contributes more than one that sells 50 times at $10 each.

### Section 4 (Final) — CTE ranking suppliers by efficiency score

In [127]:
;WITH SupplierEfficiency AS (
    SELECT
        s.SupplierId,
        s.SupplierCompanyName,
        s.SupplierCountry,
        COUNT(DISTINCT p.ProductId)                                         AS ProductsSupplied,
        COUNT(od.OrderId)                                                   AS TotalOrderLines,
        CAST(SUM(od.LineAmount) AS DECIMAL(10,2))                           AS TotalRevenue,
        CAST(SUM(od.LineAmount) / COUNT(DISTINCT p.ProductId) AS DECIMAL(10,2)) AS RevenuePerProduct
    FROM Production.Supplier s
    JOIN Production.Product p   ON s.SupplierId = p.SupplierId
    JOIN Sales.OrderDetail od   ON p.ProductId  = od.ProductId
    GROUP BY s.SupplierId, s.SupplierCompanyName, s.SupplierCountry
)
SELECT
    ROW_NUMBER() OVER (ORDER BY RevenuePerProduct ASC)  AS EfficiencyRank,
    SupplierCompanyName,
    SupplierCountry,
    ProductsSupplied,
    TotalOrderLines,
    TotalRevenue,
    RevenuePerProduct
FROM SupplierEfficiency
ORDER BY EfficiencyRank;

(29 rows affected)

EfficiencyRank | SupplierCompanyName | SupplierCountry | ProductsSupplied | TotalOrderLines | TotalRevenue | RevenuePerProduct
---------------+---------------------+-----------------+------------------+-----------------+--------------+------------------
1              | Supplier FNUXM      | Netherlands     | 2                | 27              | 5901.35      | 2950.68          
2              | Supplier UNAHG      | Brazil          | 1                | 51              | 4782.60      | 4782.60          
3              | Supplier QWUSF      | Japan           | 3                | 68              | 15678.30     | 5226.10          
4              | Supplier XOXZA      | Denmark         | 2                | 41              | 10884.50     | 5442.25          
5              | Supplier QQYEU      | Sweden          | 2                | 34              | 12072.60     | 6036.30          
6              | Supplier ZRYDZ      | France          | 1                | 18              | 6664.75      | 66

RevenuePerProduct ASC puts the least efficient suppliers at the top. These are the ones where the products they provide aren't pulling their weight. Good candidates for a supplier review meeting.


# Mystery #6 

I want to know if any customers are completely locked into buying from just one product category. These customers might be risky — if that category has a problem they might just leave. Also it could be a cross-sell opportunity.


### Section 1 — Count how many distinct categories each customer has ordered from

In [129]:
SELECT
    c.CustomerId,
    c.CustomerCompanyName,
    COUNT(DISTINCT cat.CategoryId) AS CategoriesOrdered
FROM Sales.Customer AS c
JOIN Sales.[Order] AS o        
    ON c.CustomerId = o.CustomerId
JOIN Sales.OrderDetail AS od   
    ON o.OrderId    = od.OrderId
JOIN Production.Product AS p   
    ON od.ProductId = p.ProductId
JOIN Production.Category AS cat 
    ON p.CategoryId = cat.CategoryId
GROUP BY c.CustomerId, c.CustomerCompanyName
ORDER BY CategoriesOrdered ASC;

(89 rows affected)

CustomerId | CustomerCompanyName | CategoriesOrdered
-----------+---------------------+------------------
13         | Customer VMLOG      | 2                
43         | Customer UISOJ      | 2                
53         | Customer GCJSG      | 3                
85         | Customer ENQZT      | 3                
74         | Customer YSHXL      | 4                
8          | Customer QUHWH      | 4                
33         | Customer FVXPQ      | 4                
26         | Customer USDBG      | 5                
27         | Customer WMFEA      | 5                
29         | Customer MDLWA      | 5                
1          | Customer NRZBB      | 5                
17         | Customer FEVNN      | 5                
36         | Customer LVJSO      | 5                
12         | Customer PSNMQ      | 5                
42         | Customer IAIJK      | 5                
77         | Customer LCYBZ      | 5                
78         | Customer NLTYP      | 6          

Four-table JOIN chain to get from customer all the way to category. COUNT DISTINCT on CategoryId shows how broad each customer's buying is. A value of 1 means they've only ever ordered from one category.

### Section 2 — Filter to single-category customers and show which category it is

In [ ]:
SELECT TOP 10
    c.CustomerCompanyName,
    COUNT(DISTINCT cat.CategoryId) AS CategoriesBought,
    COUNT(DISTINCT o.OrderId) AS TotalOrders,
    CAST(SUM(od.LineAmount) AS DECIMAL(10,2)) AS TotalRevenue,
    CASE
        WHEN COUNT(DISTINCT cat.CategoryId) <= 2 THEN 'Narrow buyer — cross-sell opportunity'
        WHEN COUNT(DISTINCT cat.CategoryId) <= 4 THEN 'Moderate range'
        ELSE                                          'Broad buyer'
    END AS BuyingPattern
FROM Sales.Customer AS c
JOIN Sales.[Order] AS o         
ON c.CustomerId = o.CustomerId
JOIN Sales.OrderDetail AS od    
    ON o.OrderId    = od.OrderId
JOIN Production.Product AS p    
    ON od.ProductId = p.ProductId
JOIN Production.Category AS cat 
    ON p.CategoryId = cat.CategoryId
GROUP BY c.CustomerId, c.CustomerCompanyName
ORDER BY CategoriesBought ASC, TotalRevenue DESC;

(10 rows affected)

CustomerCompanyName | CategoriesBought | TotalOrders | TotalRevenue | BuyingPattern                        
--------------------+------------------+-------------+--------------+--------------------------------------
Customer UISOJ      | 2                | 2           | 357.00       | Narrow buyer — cross-sell opportunity
Customer VMLOG      | 2                | 1           | 100.80       | Narrow buyer — cross-sell opportunity
Customer ENQZT      | 3                | 5           | 1480.00      | Moderate range                       
Customer GCJSG      | 3                | 3           | 649.00       | Moderate range                       
Customer QUHWH      | 4                | 3           | 5297.80      | Moderate range                       
Customer YSHXL      | 4                | 4           | 2423.35      | Moderate range                       
Customer FVXPQ      | 4                | 2           | 1488.70      | Moderate range                       
Customer NRZBB      | 5     

HAVING COUNT DISTINCT = 1 is the key filter. MIN(CategoryName) is a trick to pull the category name without grouping by it — since there's only one category per customer here, MIN just returns that one value.

### Section 3 — See which categories attract the most single-category customers

In [136]:
SELECT
    cat.CategoryName AS Category,
    COUNT(DISTINCT c.CustomerId) AS TotalCustomers,
    COUNT(DISTINCT o.OrderId) AS TotalOrders,
    CAST(SUM(od.LineAmount) AS DECIMAL(10,2))   AS TotalRevenue
FROM Sales.Customer AS c
JOIN Sales.[Order] AS o         
    ON c.CustomerId = o.CustomerId
JOIN Sales.OrderDetail AS od    
    ON o.OrderId    = od.OrderId
JOIN Production.Product AS p    
    ON od.ProductId = p.ProductId
JOIN Production.Category AS cat 
    ON p.CategoryId = cat.CategoryId
GROUP BY cat.CategoryId, cat.CategoryName
ORDER BY TotalCustomers DESC;

(8 rows affected)

Category       | TotalCustomers | TotalOrders | TotalRevenue
---------------+----------------+-------------+-------------
Seafood        | 85             | 291         | 141623.09   
Beverages      | 83             | 354         | 286526.95   
Dairy Products | 81             | 303         | 251330.50   
Confections    | 80             | 295         | 177099.10   
Meat/Poultry   | 69             | 161         | 178188.80   
Condiments     | 69             | 193         | 113694.75   
Grains/Cereals | 68             | 182         | 100726.80   
Produce        | 63             | 129         | 105268.60   
(8 rows)

This shows which categories are creating the most tunnel-vision customers. If Beverages has 10 customers who only ever buy beverages, that's a cross-sell opportunity — or a risk if beverages ever has supply issues.

### Section 4 (Final) — CTE showing single-category customers with cross-sell flag

In [140]:
;WITH CustomerCategories AS (
    SELECT
        c.CustomerId,
        c.CustomerCompanyName,
        COUNT(DISTINCT cat.CategoryId)          AS CategoriesBought,
        MIN(cat.CategoryName)                   AS PrimaryCategory,
        COUNT(DISTINCT o.OrderId)               AS TotalOrders,
        CAST(SUM(od.LineAmount) AS DECIMAL(10,2)) AS TotalRevenue
    FROM Sales.Customer c
    JOIN Sales.[Order] o         ON c.CustomerId = o.CustomerId
    JOIN Sales.OrderDetail od    ON o.OrderId    = od.OrderId
    JOIN Production.Product p    ON od.ProductId = p.ProductId
    JOIN Production.Category cat ON p.CategoryId = cat.CategoryId
    GROUP BY c.CustomerId, c.CustomerCompanyName
)
SELECT TOP 10
    CustomerCompanyName,
    CategoriesBought,
    PrimaryCategory,
    TotalOrders,
    TotalRevenue,
    CASE
        WHEN CategoriesBought <= 2 AND TotalRevenue > 5000
            THEN 'High value — cross-sell opportunity'
        WHEN CategoriesBought <= 2
            THEN 'Narrow buyer — worth monitoring'
        ELSE 'Multi-category buyer'
    END AS CustomerStatus
FROM CustomerCategories
ORDER BY CategoriesBought ASC, TotalRevenue DESC;

(10 rows affected)

CustomerCompanyName | CategoriesBought | PrimaryCategory | TotalOrders | TotalRevenue | CustomerStatus                 
--------------------+------------------+-----------------+-------------+--------------+--------------------------------
Customer UISOJ      | 2                | Dairy Products  | 2           | 357.00       | Narrow buyer — worth monitoring
Customer VMLOG      | 2                | Confections     | 1           | 100.80       | Narrow buyer — worth monitoring
Customer ENQZT      | 3                | Dairy Products  | 5           | 1480.00      | Multi-category buyer           
Customer GCJSG      | 3                | Beverages       | 3           | 649.00       | Multi-category buyer           
Customer QUHWH      | 4                | Beverages       | 3           | 5297.80      | Multi-category buyer           
Customer YSHXL      | 4                | Beverages       | 4           | 2423.35      | Multi-category buyer           
Customer FVXPQ      | 4                |

The CTE cleans up the aggregation and then I apply a CASE in the outer query.


# Mystery #7 

Customers complain about late deliveries but nobody knows which shipper is the real problem. I want to look at how many orders each shipper delivered within a reasonable window versus how many went over. That gives us an actual on-time rate.

### Section 1 — Calculate delivery days for every order

In [142]:
SELECT
    o.OrderId,
    o.ShipperId,
    o.OrderDate,
    o.ShipToDate,
    DATEDIFF(DAY, o.OrderDate, o.ShipToDate)    AS DeliveryDays
FROM Sales.[Order] AS o
WHERE o.ShipToDate IS NOT NULL
ORDER BY DeliveryDays DESC;

(810 rows affected)

OrderId | ShipperId | OrderDate  | ShipToDate | DeliveryDays
--------+-----------+------------+------------+-------------
10660   | 1         | 2021-09-08 | 2021-10-15 | 37          
10777   | 2         | 2021-12-15 | 2022-01-21 | 37          
10545   | 2         | 2021-05-22 | 2021-06-26 | 35          
10593   | 2         | 2021-07-09 | 2021-08-13 | 35          
10380   | 3         | 2020-12-12 | 2021-01-16 | 35          
10427   | 2         | 2021-01-27 | 2021-03-03 | 35          
10924   | 2         | 2022-03-04 | 2022-04-08 | 35          
10927   | 1         | 2022-03-05 | 2022-04-08 | 34          
10309   | 1         | 2020-09-19 | 2020-10-23 | 34          
10705   | 2         | 2021-10-15 | 2021-11-18 | 34          
10709   | 3         | 2021-10-17 | 2021-11-20 | 34          
10726   | 1         | 2021-11-03 | 2021-12-05 | 32          
10727   | 1         | 2021-11-03 | 2021-12-05 | 32          
10596   | 1         | 2021-07-11 | 2021-08-12 | 32          
10366   | 2         | 20

DATEDIFF between OrderDate and ShipToDate gives delivery days. I am using ShipToDate as the delivery date as when it actually arrived. Filtering out NULLs removes unshipped orders.

### Section 2 — Tag each order as on-time or late, then group by shipper

In [143]:
SELECT
    s.ShipperCompanyName,
    COUNT(o.OrderId) AS TotalOrders,
    SUM(CASE WHEN DATEDIFF(DAY, o.OrderDate, o.ShipToDate) <= 10
             THEN 1 ELSE 0 END)                     AS OnTimeOrders,
    SUM(CASE WHEN DATEDIFF(DAY, o.OrderDate, o.ShipToDate) > 10
             THEN 1 ELSE 0 END)                     AS LateOrders
FROM Sales.Shipper s
JOIN Sales.[Order] o ON s.ShipperId = o.ShipperId
WHERE o.ShipToDate IS NOT NULL
GROUP BY s.ShipperId, s.ShipperCompanyName
ORDER BY LateOrders DESC;

(3 rows affected)

ShipperCompanyName | TotalOrders | OnTimeOrders | LateOrders
-------------------+-------------+--------------+-----------
Shipper ETYNR      | 315         | 248          | 67        
Shipper GVSUA      | 246         | 202          | 44        
Shipper ZHISN      | 249         | 218          | 31        
(3 rows)

 It lets me count only the rows that meet a condition without needing a subquery. One pass through the data gives me both on-time and late counts at once.

### Section 3 — Calculate the on-time percentage per shipper

In [144]:
SELECT
    s.ShipperCompanyName,
    COUNT(o.OrderId)                                        AS TotalOrders,
    SUM(CASE WHEN DATEDIFF(DAY, o.OrderDate, o.ShipToDate) <= 10
             THEN 1 ELSE 0 END)                             AS OnTime,
    CAST(
        CAST(SUM(CASE WHEN DATEDIFF(DAY, o.OrderDate, o.ShipToDate) <= 10
                      THEN 1 ELSE 0 END) AS FLOAT)
        / COUNT(o.OrderId) * 100
    AS DECIMAL(10,1))                                       AS OnTimePct,
    CAST(AVG(DATEDIFF(DAY, o.OrderDate, o.ShipToDate)) AS DECIMAL(10,1)) AS AvgDeliveryDays
FROM Sales.Shipper s
JOIN Sales.[Order] o ON s.ShipperId = o.ShipperId
WHERE o.ShipToDate IS NOT NULL
GROUP BY s.ShipperId, s.ShipperCompanyName
ORDER BY OnTimePct ASC;

(3 rows affected)

ShipperCompanyName | TotalOrders | OnTime | OnTimePct | AvgDeliveryDays
-------------------+-------------+--------+-----------+----------------
Shipper ETYNR      | 315         | 248    | 78.7      | 9.0            
Shipper GVSUA      | 246         | 202    | 82.1      | 8.0            
Shipper ZHISN      | 249         | 218    | 87.6      | 7.0            
(3 rows)

### Section 4 (Final) — CTE with performance grade

In [145]:
;WITH ShipperStats AS (
    SELECT
        s.ShipperCompanyName,
        COUNT(o.OrderId) AS TotalOrders,
        SUM(CASE WHEN DATEDIFF(DAY, o.OrderDate, o.ShipToDate) <= 10
                 THEN 1 ELSE 0 END) AS OnTime,
        CAST(
            CAST(SUM(CASE WHEN DATEDIFF(DAY, o.OrderDate, o.ShipToDate) <= 10
                          THEN 1 ELSE 0 END) AS FLOAT)
            / COUNT(o.OrderId) * 100
        AS DECIMAL(10,1))  AS OnTimePct,
        CAST(AVG(DATEDIFF(DAY, o.OrderDate, o.ShipToDate))
        AS DECIMAL(10,1)) AS AvgDeliveryDays
    FROM Sales.Shipper s
    JOIN Sales.[Order] o ON s.ShipperId = o.ShipperId
    WHERE o.ShipToDate IS NOT NULL
    GROUP BY s.ShipperId, s.ShipperCompanyName
)
SELECT
    ROW_NUMBER() OVER (ORDER BY OnTimePct DESC) AS ReliabilityRank,
    ShipperCompanyName,
    TotalOrders,
    OnTime,
    OnTimePct,
    AvgDeliveryDays,
    CASE
        WHEN OnTimePct >= 80 THEN 'Good'
        WHEN OnTimePct >= 60 THEN 'Average'
        ELSE 'Poor'
    END AS PerformanceGrade
FROM ShipperStats
ORDER BY ReliabilityRank;

(3 rows affected)

ReliabilityRank | ShipperCompanyName | TotalOrders | OnTime | OnTimePct | AvgDeliveryDays | PerformanceGrade
----------------+--------------------+-------------+--------+-----------+-----------------+-----------------
1               | Shipper ZHISN      | 249         | 218    | 87.6      | 7.0             | Good            
2               | Shipper GVSUA      | 246         | 202    | 82.1      | 8.0             | Good            
3               | Shipper ETYNR      | 315         | 248    | 78.7      | 9.0             | Average         
(3 rows)

The grade makes it easy to communicate to management without them having to interpret the raw percentage. Found each shipper's on-time delivery rate and ranked them by reliability. The lowest ranked shipper is where the customer complaints are probably coming from.


# Mystery #8 

I want to check if any employees are basically just managing one or two big accounts and not really developing new relationships. Also I want to see the management chain above them if someone low in the org chart has a narrow customer base, is their manager aware?


### Section 1 — Count unique customers per employee

In [146]:
SELECT
    e.EmployeeId,
    e.EmployeeFirstName + ' ' + e.EmployeeLastName  AS EmployeeName,
    e.EmployeeTitle,
    COUNT(DISTINCT o.CustomerId) AS UniqueCustomers,
    COUNT(DISTINCT o.OrderId) AS TotalOrders
FROM HumanResources.Employee e
JOIN Sales.[Order] AS o 
    ON e.EmployeeId = o.EmployeeId
GROUP BY e.EmployeeId, e.EmployeeFirstName, e.EmployeeLastName, e.EmployeeTitle
ORDER BY UniqueCustomers ASC;

(9 rows affected)

EmployeeId | EmployeeName   | EmployeeTitle         | UniqueCustomers | TotalOrders
-----------+----------------+-----------------------+-----------------+------------
5          | Sven Mortensen | Sales Manager         | 29              | 42         
9          | Patricia Doyle | Sales Representative  | 29              | 43         
6          | Paul Suurs     | Sales Representative  | 43              | 67         
7          | Russell King   | Sales Representative  | 45              | 72         
8          | Maria Cameron  | Sales Representative  | 56              | 104        
2          | Don Funk       | Vice President, Sales | 59              | 96         
3          | Judy Lew       | Sales Manager         | 63              | 127        
1          | Sara Davis     | CEO                   | 65              | 124        
4          | Yael Peled     | Sales Representative  | 75              | 156        
(9 rows)

COUNT DISTINCT on CustomerId gives me unique customers per employee. Comparing that to TotalOrders shows me the ratio an employee with 5 customers but 200 orders is serving the same accounts over and over.

### Section 2 — Build the management hierarchy using a Recursive CTE

In [148]:
WITH ManagementChain AS (
    SELECT
        e.EmployeeId,
        e.EmployeeFirstName + ' ' + e.EmployeeLastName AS EmployeeName,
        e.EmployeeTitle,
        e.EmployeeManagerId,
        CAST(e.EmployeeFirstName + ' ' + e.EmployeeLastName
            AS VARCHAR(500)) AS ReportingChain,
        0 AS OrgLevel
    FROM HumanResources.Employee e
    WHERE e.EmployeeManagerId IS NULL

    UNION ALL

    SELECT
        e.EmployeeId,
        e.EmployeeFirstName + ' ' + e.EmployeeLastName,
        e.EmployeeTitle,
        e.EmployeeManagerId,
        CAST(mc.ReportingChain + ' > ' +
             e.EmployeeFirstName + ' ' + e.EmployeeLastName
            AS VARCHAR(500)),
        mc.OrgLevel + 1
    FROM HumanResources.Employee e
    JOIN ManagementChain AS mc 
        ON e.EmployeeManagerId = mc.EmployeeId
)
SELECT
    OrgLevel AS HierarchyLevel,
    EmployeeName,
    EmployeeTitle,
    ReportingChain
FROM ManagementChain
ORDER BY OrgLevel, EmployeeName;

(9 rows affected)

HierarchyLevel | EmployeeName   | EmployeeTitle         | ReportingChain                                         
---------------+----------------+-----------------------+--------------------------------------------------------
0              | Sara Davis     | CEO                   | Sara Davis                                             
1              | Don Funk       | Vice President, Sales | Sara Davis > Don Funk                                  
2              | Judy Lew       | Sales Manager         | Sara Davis > Don Funk > Judy Lew                       
2              | Sven Mortensen | Sales Manager         | Sara Davis > Don Funk > Sven Mortensen                 
3              | Maria Cameron  | Sales Representative  | Sara Davis > Don Funk > Judy Lew > Maria Cameron       
3              | Patricia Doyle | Sales Representative  | Sara Davis > Don Funk > Sven Mortensen > Patricia Doyle
3              | Paul Suurs     | Sales Representative  | Sara Davis > Don Funk > Sven M

The recursive CTE works in two parts the anchor gets the top-level people and the recursive member keeps joining one level deeper until it runs out of employees to add. 

### Section 3 — Calculate the ratio of orders to unique customers per employee

In [149]:
SELECT
    e.EmployeeFirstName + ' ' + e.EmployeeLastName AS EmployeeName,
    e.EmployeeTitle,
    COUNT(DISTINCT o.CustomerId) AS UniqueCustomers,
    COUNT(DISTINCT o.OrderId) AS TotalOrders,
    CAST(
        CAST(COUNT(DISTINCT o.OrderId) AS FLOAT)
        / NULLIF(COUNT(DISTINCT o.CustomerId), 0)
    AS DECIMAL(10,1)) AS OrdersPerCustomer
FROM HumanResources.Employee e
JOIN Sales.[Order] AS o 
    ON e.EmployeeId = o.EmployeeId
GROUP BY e.EmployeeId, e.EmployeeFirstName, e.EmployeeLastName, e.EmployeeTitle
ORDER BY OrdersPerCustomer DESC;

(9 rows affected)

EmployeeName   | EmployeeTitle         | UniqueCustomers | TotalOrders | OrdersPerCustomer
---------------+-----------------------+-----------------+-------------+------------------
Yael Peled     | Sales Representative  | 75              | 156         | 2.1              
Judy Lew       | Sales Manager         | 63              | 127         | 2.0              
Sara Davis     | CEO                   | 65              | 124         | 1.9              
Maria Cameron  | Sales Representative  | 56              | 104         | 1.9              
Paul Suurs     | Sales Representative  | 43              | 67          | 1.6              
Russell King   | Sales Representative  | 45              | 72          | 1.6              
Don Funk       | Vice President, Sales | 59              | 96          | 1.6              
Patricia Doyle | Sales Representative  | 29              | 43          | 1.5              
Sven Mortensen | Sales Manager         | 29              | 42          | 1.4              

NULLIF prevents divide by zero if somehow an employee has 0 customers. A high number here means repeat business from the same clients not necessarily bad but worth knowing.

### Section 4 (Final) — Combine org chart with customer diversity metrics

In [150]:
;WITH ManagementChain AS (
    SELECT
        e.EmployeeId,
        e.EmployeeFirstName + ' ' + e.EmployeeLastName AS EmployeeName,
        e.EmployeeManagerId,
        CAST(e.EmployeeFirstName + ' ' + e.EmployeeLastName
            AS VARCHAR(500)) AS ReportingChain,
        0 AS OrgLevel
    FROM HumanResources.Employee e
    WHERE e.EmployeeManagerId IS NULL

    UNION ALL

    SELECT
        e.EmployeeId,
        e.EmployeeFirstName + ' ' + e.EmployeeLastName,
        e.EmployeeManagerId,
        CAST(mc.ReportingChain + ' > ' +
             e.EmployeeFirstName + ' ' + e.EmployeeLastName
            AS VARCHAR(500)),
        mc.OrgLevel + 1
    FROM HumanResources.Employee e
    JOIN ManagementChain mc ON e.EmployeeManagerId = mc.EmployeeId
),
CustomerMetrics AS (
    SELECT
        o.EmployeeId,
        COUNT(DISTINCT o.CustomerId) AS UniqueCustomers,
        COUNT(DISTINCT o.OrderId)  AS TotalOrders,
        CAST(
            CAST(COUNT(DISTINCT o.OrderId) AS FLOAT)
            / NULLIF(COUNT(DISTINCT o.CustomerId), 0)
        AS DECIMAL(10,1)) AS OrdersPerCustomer
    FROM Sales.[Order] AS o
    GROUP BY o.EmployeeId
)
SELECT
    mc.OrgLevel,
    mc.ReportingChain,
    cm.UniqueCustomers,
    cm.TotalOrders,
    cm.OrdersPerCustomer,
    CASE
        WHEN cm.UniqueCustomers < 5  THEN 'Narrow — review account spread'
        WHEN cm.UniqueCustomers < 15 THEN 'Moderate'
        ELSE                              'Broad customer base'
    END AS AccountDiversity
FROM CustomerMetrics AS cm
JOIN ManagementChain AS mc 
    ON cm.EmployeeId = mc.EmployeeId
ORDER BY cm.UniqueCustomers ASC;

(9 rows affected)

OrgLevel | ReportingChain                                          | UniqueCustomers | TotalOrders | OrdersPerCustomer | AccountDiversity   
---------+---------------------------------------------------------+-----------------+-------------+-------------------+--------------------
2        | Sara Davis > Don Funk > Sven Mortensen                  | 29              | 42          | 1.4               | Broad customer base
3        | Sara Davis > Don Funk > Sven Mortensen > Patricia Doyle | 29              | 43          | 1.5               | Broad customer base
3        | Sara Davis > Don Funk > Sven Mortensen > Paul Suurs     | 43              | 67          | 1.6               | Broad customer base
3        | Sara Davis > Don Funk > Sven Mortensen > Russell King   | 45              | 72          | 1.6               | Broad customer base
3        | Sara Davis > Don Funk > Judy Lew > Maria Cameron        | 56              | 104         | 1.9               | Broad customer base
1        | Sa

Two CTEs chained together ManagementChain gives the org structure, CustomerMetrics gives the workload numbers. Joining them on EmployeeId shows both at once. The ReportingChain column shows who you'd need to escalate to if an employee's account spread is too narrow.


# Mystery #9 

I want to find which categories have the biggest spread between their cheapest and most expensive products.

### Section 1 — Find the min, max and average price per category

In [151]:
SELECT
    c.CategoryName,
    COUNT(p.ProductId) AS ProductCount,
    MIN(p.UnitPrice) AS CheapestProduct,
    MAX(p.UnitPrice) AS MostExpensive,
    CAST(AVG(p.UnitPrice) AS DECIMAL(10,2)) AS AvgPrice
FROM Production.Category AS c
JOIN Production.Product AS p 
    ON c.CategoryId = p.CategoryId
GROUP BY c.CategoryId, c.CategoryName
ORDER BY MostExpensive DESC;

(8 rows affected)

CategoryName   | ProductCount | CheapestProduct | MostExpensive | AvgPrice
---------------+--------------+-----------------+---------------+---------
Beverages      | 12           | 4.50            | 263.50        | 37.98   
Meat/Poultry   | 6            | 7.45            | 123.79        | 54.01   
Confections    | 13           | 9.20            | 81.00         | 25.16   
Seafood        | 12           | 6.00            | 62.50         | 20.68   
Dairy Products | 10           | 2.50            | 55.00         | 28.73   
Produce        | 5            | 10.00           | 53.00         | 32.37   
Condiments     | 12           | 10.00           | 43.90         | 23.06   
Grains/Cereals | 7            | 7.00            | 38.00         | 20.25   
(8 rows)

 MIN and MAX on UnitPrice gives me the cheapest and most expensive product in each category. This is from the catalog, not from actual orders, which feels more appropriate for a pricing spread question.

### Section 2 — Calculate the price range and coefficient of variation

In [ ]:
SELECT
    c.CategoryName,
    COUNT(p.ProductId) AS ProductCount,
    CAST(MIN(p.UnitPrice) AS DECIMAL(10,2)) AS MinPrice,
    CAST(MAX(p.UnitPrice) AS DECIMAL(10,2)) AS MaxPrice,
    CAST(MAX(p.UnitPrice) - MIN(p.UnitPrice) AS DECIMAL(10,2)) AS PriceRange,
    CAST(AVG(p.UnitPrice) AS DECIMAL(10,2)) AS AvgPrice,
    CAST(STDEV(p.UnitPrice) AS DECIMAL(10,2)) AS PriceStdDev
FROM Production.Category AS c
JOIN Production.Product AS p 
    ON c.CategoryId = p.CategoryId
GROUP BY c.CategoryId, c.CategoryName
ORDER BY PriceRange DESC;

(8 rows affected)

CategoryName   | ProductCount | MinPrice | MaxPrice | PriceRange | AvgPrice | PriceStdDev
---------------+--------------+----------+----------+------------+----------+------------
Beverages      | 12           | 4.50     | 263.50   | 259.00     | 37.98    | 71.73      
Meat/Poultry   | 6            | 7.45     | 123.79   | 116.34     | 54.01    | 45.74      
Confections    | 13           | 9.20     | 81.00    | 71.80      | 25.16    | 21.26      
Seafood        | 12           | 6.00     | 62.50    | 56.50      | 20.68    | 15.21      
Dairy Products | 10           | 2.50     | 55.00    | 52.50      | 28.73    | 14.79      
Produce        | 5            | 10.00    | 53.00    | 43.00      | 32.37    | 17.25      
Condiments     | 12           | 10.00    | 43.90    | 33.90      | 23.06    | 10.19      
Grains/Cereals | 7            | 7.00     | 38.00    | 31.00      | 20.25    | 11.74      
(8 rows)

### Section 3 — Show the actual cheapest and most expensive product per category

In [155]:
SELECT
    c.CategoryName,
    p.ProductName,
    p.UnitPrice,
    CASE
        WHEN p.UnitPrice = MIN(p.UnitPrice) OVER (PARTITION BY c.CategoryId)
            THEN 'Cheapest in category'
        WHEN p.UnitPrice = MAX(p.UnitPrice) OVER (PARTITION BY c.CategoryId)
            THEN 'Most expensive in category'
    END AS PricePosition
FROM Production.Category AS c
JOIN Production.Product AS p 
    ON c.CategoryId = p.CategoryId
WHERE
    p.UnitPrice = (SELECT MIN(UnitPrice) FROM Production.Product WHERE CategoryId = c.CategoryId)
    OR p.UnitPrice = (SELECT MAX(UnitPrice) FROM Production.Product WHERE CategoryId = c.CategoryId)
ORDER BY c.CategoryName, p.UnitPrice;

(16 rows affected)

CategoryName   | ProductName   | UnitPrice | PricePosition             
---------------+---------------+-----------+---------------------------
Beverages      | Product QOGNU | 4.50      | Cheapest in category      
Beverages      | Product QDOMO | 263.50    | Most expensive in category
Condiments     | Product IMEHJ | 10.00     | Cheapest in category      
Condiments     | Product ICKNK | 43.90     | Most expensive in category
Confections    | Product XKXDO | 9.20      | Cheapest in category      
Confections    | Product QHFFP | 81.00     | Most expensive in category
Dairy Products | Product ASTMN | 2.50      | Cheapest in category      
Dairy Products | Product UKXRI | 55.00     | Most expensive in category
Grains/Cereals | Product QSRXF | 7.00      | Cheapest in category      
Grains/Cereals | Product VKCMF | 38.00     | Most expensive in category
Meat/Poultry   | Product QAQRL | 7.45      | Cheapest in category      
Meat/Poultry   | Product VJXYN | 123.79    | Most expensive in c

I want to actually name the cheapest and most expensive product in each category, not just show numbers. The WHERE clause with subqueries filters to only those boundary products. The CASE then labels them clearly.

### Section 4 (Final) — CTE ranking categories by price spread

In [ ]:
;WITH CategoryPricing AS (
    SELECT
        c.CategoryId,
        c.CategoryName,
        COUNT(p.ProductId) AS ProductCount,
        CAST(MIN(p.UnitPrice) AS DECIMAL(10,2)) AS MinPrice,
        CAST(MAX(p.UnitPrice) AS DECIMAL(10,2)) AS MaxPrice,
        CAST(MAX(p.UnitPrice) - MIN(p.UnitPrice) AS DECIMAL(10,2)) AS PriceRange,
        CAST(AVG(p.UnitPrice) AS DECIMAL(10,2)) AS AvgPrice,
        CAST(STDEV(p.UnitPrice) AS DECIMAL(10,2)) AS PriceStdDev
    FROM Production.Category AS c
    JOIN Production.Product AS p 
        ON c.CategoryId = p.CategoryId
    GROUP BY c.CategoryId, c.CategoryName
)
SELECT
    ROW_NUMBER() OVER (ORDER BY PriceRange DESC)    AS SpreadRank,
    CategoryName,
    ProductCount,
    MinPrice,
    MaxPrice,
    PriceRange,
    AvgPrice,
    PriceStdDev,
    CASE
        WHEN PriceRange > AvgPrice * 2  THEN 'Very inconsistent pricing'
        WHEN PriceRange > AvgPrice      THEN 'Some spread'
        ELSE                                 'Relatively consistent'
    END AS PricingNote
FROM CategoryPricing
ORDER BY SpreadRank;

(8 rows affected)

SpreadRank | CategoryName   | ProductCount | MinPrice | MaxPrice | PriceRange | AvgPrice | PriceStdDev | PricingNote              
-----------+----------------+--------------+----------+----------+------------+----------+-------------+--------------------------
1          | Beverages      | 12           | 4.50     | 263.50   | 259.00     | 37.98    | 71.73       | Very inconsistent pricing
2          | Meat/Poultry   | 6            | 7.45     | 123.79   | 116.34     | 54.01    | 45.74       | Very inconsistent pricing
3          | Confections    | 13           | 9.20     | 81.00    | 71.80      | 25.16    | 21.26       | Very inconsistent pricing
4          | Seafood        | 12           | 6.00     | 62.50    | 56.50      | 20.68    | 15.21       | Very inconsistent pricing
5          | Dairy Products | 10           | 2.50     | 55.00    | 52.50      | 28.73    | 14.79       | Some spread              
6          | Produce        | 5            | 10.00    | 53.00    | 43.00      | 32.

We Found which categories have the most inconsistent pricing across their products, ranked from biggest to smallest spread.


# Mystery #10 

I want to find customers whose first order was their biggest, and who barely ordered again after that. These are probably lost customers we could win back.


### Section 1 — Find each customer's first order date and total order count

In [157]:
SELECT
    c.CustomerId,
    c.CustomerCompanyName,
    MIN(o.OrderDate) AS FirstOrderDate,
    MAX(o.OrderDate) AS LastOrderDate,
    COUNT(o.OrderId) AS TotalOrders
FROM Sales.Customer AS c
JOIN Sales.[Order] AS o 
    ON c.CustomerId = o.CustomerId
GROUP BY c.CustomerId, c.CustomerCompanyName
ORDER BY TotalOrders ASC;

(89 rows affected)

CustomerId | CustomerCompanyName | FirstOrderDate | LastOrderDate | TotalOrders
-----------+---------------------+----------------+---------------+------------
13         | Customer VMLOG      | 2020-07-18     | 2020-07-18    | 1          
33         | Customer FVXPQ      | 2020-07-30     | 2021-12-18    | 2          
43         | Customer UISOJ      | 2021-03-21     | 2021-05-22    | 2          
42         | Customer IAIJK      | 2021-04-03     | 2022-01-01    | 3          
53         | Customer GCJSG      | 2021-04-24     | 2022-04-29    | 3          
26         | Customer USDBG      | 2021-09-17     | 2022-03-24    | 3          
16         | Customer GYBBY      | 2021-02-04     | 2022-01-23    | 3          
8          | Customer QUHWH      | 2020-10-10     | 2022-03-24    | 3          
78         | Customer NLTYP      | 2021-08-07     | 2022-04-06    | 3          
82         | Customer EYHKM      | 2021-06-19     | 2022-01-08    | 3          
77         | Customer LCYBZ      | 2020-

MIN and MAX of OrderDate tell me when a customer first and last ordered. Total order count shows how active they've been overall. Customers with TotalOrders = 1 or 2 and an early FirstOrderDate are my main targets.

### Section 2 — Calculate the value of each customer's first order

In [ ]:
SELECT
    c.CustomerCompanyName,
    o.OrderDate AS FirstOrderDate,
    ROUND(SUM(od.LineAmount), 2) AS FirstOrderValue
FROM Sales.Customer AS c
JOIN Sales.[Order] AS o 
    ON c.CustomerId = o.CustomerId
JOIN Sales.OrderDetail AS od 
    ON o.OrderId = od.OrderId
WHERE o.OrderDate = (
    SELECT MIN(o2.OrderDate)
    FROM Sales.[Order] AS o2
    WHERE o2.CustomerId = c.CustomerId
)
GROUP BY c.CustomerId, c.CustomerCompanyName, o.OrderDate
ORDER BY FirstOrderValue DESC;

(89 rows affected)

CustomerCompanyName | FirstOrderDate | FirstOrderValue
--------------------+----------------+----------------
Customer WFIZJ      | 2020-12-04     | 12281.20       
Customer LOLJO      | 2020-11-13     | 10741.60       
Customer LCOUJ      | 2020-10-08     | 6155.90        
Customer KZQZT      | 2020-09-13     | 4157.00        
Customer AZJED      | 2020-07-29     | 4031.00        
Customer SFOGW      | 2020-07-09     | 3730.00        
Customer AHPOP      | 2020-11-21     | 3654.40        
Customer FRXZL      | 2020-09-05     | 3127.00        
Customer CCKOT      | 2020-07-12     | 2490.50        
Customer PVDZC      | 2020-10-17     | 2233.60        
Customer JUWXK      | 2020-08-27     | 2169.00        
Customer IRRVL      | 2020-08-05     | 2142.40        
Customer THHDP      | 2020-07-17     | 2018.60        
Customer FAPSM      | 2020-07-05     | 1863.40        
Customer EEALV      | 2020-12-20     | 1832.80        
Customer IBVRG      | 2020-07-08     | 1813.00        
Customer Q

The correlated subquery in WHERE finds the first order date for each customer and then I filter to only that order. Now I have the first order's value isolated for every customer.

### Section 3 — Compare first order value to total lifetime revenue

In [183]:
WITH FirstOrder AS (
    SELECT
        o.CustomerId,
        CAST(SUM(od.LineAmount) AS DECIMAL(10,2))   AS FirstOrderValue
    FROM Sales.[Order] o
    JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
    WHERE o.OrderDate = (
        SELECT MIN(o2.OrderDate)
        FROM Sales.[Order] o2
        WHERE o2.CustomerId = o.CustomerId
    )
    GROUP BY o.CustomerId
),
LifetimeRevenue AS (
    SELECT
        o.CustomerId,
        COUNT(DISTINCT o.OrderId)                   AS TotalOrders,
        CAST(SUM(od.LineAmount) AS DECIMAL(10,2))   AS LifetimeRevenue
    FROM Sales.[Order] o
    JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
    GROUP BY o.CustomerId
)
SELECT
    c.CustomerCompanyName,
    lr.TotalOrders,
    lr.LifetimeRevenue,
    fo.FirstOrderValue,
    CAST(
        fo.FirstOrderValue / NULLIF(lr.LifetimeRevenue, 0) * 100
    AS DECIMAL(10,1))                               AS FirstOrderPct
FROM Sales.Customer c
JOIN FirstOrder fo      ON c.CustomerId = fo.CustomerId
JOIN LifetimeRevenue lr ON c.CustomerId = lr.CustomerId
ORDER BY FirstOrderPct DESC;

(89 rows affected)

CustomerCompanyName | TotalOrders | LifetimeRevenue | FirstOrderValue | FirstOrderPct
--------------------+-------------+-----------------+-----------------+--------------
Customer VMLOG      | 1           | 100.80          | 100.80          | 100.0        
Customer FVXPQ      | 2           | 1488.70         | 1101.20         | 74.0         
Customer NLTYP      | 3           | 1947.24         | 1393.24         | 71.5         
Customer JUWXK      | 5           | 3810.75         | 2169.00         | 56.9         
Customer GCJSG      | 3           | 649.00          | 352.00          | 54.2         
Customer IAIJK      | 3           | 522.50          | 278.00          | 53.2         
Customer EYHKM      | 3           | 1571.20         | 764.30          | 48.6         
Customer UISOJ      | 2           | 357.00          | 147.00          | 41.2         
Customer LOLJO      | 10          | 26259.95        | 10741.60        | 40.9         
Customer WFIZJ      | 13          | 30226.10        | 

I can compare FirstOrderValue to LifetimeRevenue to see how much of their total came from that first purchase.

### Section 4 (Final) — CTE identifying one-hit-wonder customers

In [ ]:
;WITH FirstOrders AS (
    SELECT
        o.CustomerId,
        o.OrderDate AS FirstOrderDate,
        CAST(SUM(od.LineAmount) AS DECIMAL(10,2)) AS FirstOrderValue
    FROM Sales.[Order] o
    JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
    WHERE o.OrderDate = (
        SELECT MIN(o2.OrderDate)
        FROM Sales.[Order] o2
        WHERE o2.CustomerId = o.CustomerId
    )
    GROUP BY o.CustomerId, o.OrderDate
),
LifetimeTotals AS (
    SELECT
        o.CustomerId,
        COUNT(DISTINCT o.OrderId) AS TotalOrders,
        CAST(SUM(od.LineAmount) AS DECIMAL(10,2)) AS LifetimeRevenue,
        MAX(o.OrderDate) AS LastOrderDate
    FROM Sales.[Order] o
    JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
    GROUP BY o.CustomerId
)
SELECT
    c.CustomerCompanyName,
    fo.FirstOrderDate,
    fo.FirstOrderValue,
    lt.TotalOrders,
    lt.LifetimeRevenue,
    lt.LastOrderDate,
    CAST(fo.FirstOrderValue / NULLIF(lt.LifetimeRevenue, 0) * 100
        AS DECIMAL(10,1))                                   AS FirstOrderPct,
    CASE
        WHEN fo.FirstOrderValue / NULLIF(lt.LifetimeRevenue, 0) > 0.7
             AND lt.TotalOrders <= 3
            THEN 'One-hit wonder — win back target'
        WHEN lt.TotalOrders <= 2
            THEN 'Low activity — at risk'
        ELSE 'Regular customer'
    END AS CustomerStatus
FROM Sales.Customer AS c
JOIN FirstOrders AS fo    
    ON c.CustomerId = fo.CustomerId
JOIN LifetimeTotals AS lt 
    ON c.CustomerId = lt.CustomerId
ORDER BY FirstOrderPct DESC;

(89 rows affected)

CustomerCompanyName | FirstOrderDate | FirstOrderValue | TotalOrders | LifetimeRevenue | LastOrderDate | FirstOrderPct | CustomerStatus                  
--------------------+----------------+-----------------+-------------+-----------------+---------------+---------------+---------------------------------
Customer VMLOG      | 2020-07-18     | 100.80          | 1           | 100.80          | 2020-07-18    | 100.0         | One-hit wonder — win back target
Customer FVXPQ      | 2020-07-30     | 1101.20         | 2           | 1488.70         | 2021-12-18    | 74.0          | One-hit wonder — win back target
Customer NLTYP      | 2021-08-07     | 1393.24         | 3           | 1947.24         | 2022-04-06    | 71.5          | One-hit wonder — win back target
Customer JUWXK      | 2020-08-27     | 2169.00         | 5           | 3810.75         | 2022-04-22    | 56.9          | Regular customer                
Customer GCJSG      | 2021-04-24     | 352.00          | 3           | 649.0

 Two CTEs — FirstOrders gets the first purchase value, LifetimeTotals gets everything else. Joining them gives me both on one row per customer. FirstOrderPct shows what fraction of their total spend happened on day one. Customers at 70%+ with only 1-3 total orders are the win-back targets. We Found customers who came in with a big first order and then basically disappeared. These are the highest priority for a win-back campaign.